# WellClass to GaP: build an LGR grid

This is the end-to-end recipe. It converts the legacy Wildcat CSV into the current `hole_casings` schema, processes it with WellClass, adapts the derived records to GaP frames, and writes a CARFIN/LGR GRDECL using the matching Wildcat grid.

In [ ]:
from pathlib import Path
import tempfile
from src.WellClass.libs.utils.csv_parser import csv_parser
from src.WellClass.libs.well_class import WellProcessed
from src.WellClass.libs.grid_utils import WellDataFrame, LGRBuilder

root = Path.cwd()
if not (root / 'test_data').exists():
    root = root.parent
csv_path = root / 'test_data/examples/wildcat/GaP_input_Wildcat_v3.csv'
grid_case = root / 'test_data/examples/wildcat/model/TEMP-0'

def records(table):
    count = len(next(iter(table.values())))
    return [{key: values[index] for key, values in table.items()} for index in range(count)]

## Convert input and process the well

In [ ]:
legacy = csv_parser(csv_path)
header = legacy['well_header']
well_header = {
    'unique_wellbore_identifier': header['well_name'],
    'depth_reference_rkb': float(header['well_rkb']),
    'depth_reference_rkb_unit': 'm',
    'ground_elevation': float(header['sf_depth_msl']),
    'ground_elevation_unit': 'm',
    'total_depth_rkb': float(header['well_td_rkb']),
    'total_depth_rkb_unit': 'm',
}
holes = [dict(row, name=f'Hole {row["diameter_in"]} in', type='hole') for row in records(legacy['drilling'])]
casings, casing_cement = [], []
for row in records(legacy['casing_cement']):
    casings.append({**row, 'name': f'Casing {row["diameter_in"]} in', 'type': 'casing'})
    casing_cement.append({
        'name': f'Cement {row["diameter_in"]} in',
        'type': 'casing cement',
        'top_rkb': row['toc_rkb'],
        'bottom_rkb': row['boc_rkb'],
        'diameter_in': row['diameter_in'],
    })
processed_well = WellProcessed(header=well_header, hole_casings=holes + casings + casing_cement)
well_frames = WellDataFrame(processed_well, oh_perm=10000.0, cb_perm=0.05, barrier_perm=0.05)
print('holes:', len(well_frames.holes_df), 'casings:', len(well_frames.casings_df), 'casing cement:', len(processed_well.casing_cement))

## Build the GaP LGR output

In [ ]:
builder = LGRBuilder(str(grid_case), well_frames.annulus_df, well_frames.holes_df, False)
with tempfile.TemporaryDirectory() as output_dir:
    gap_casing = builder.build_grdecl(
        output_dir,
        'SCREEN_LGR',
        well_frames.holes_df,
        well_frames.casings_df,
        well_frames.barrier_regions_df,
    )
    output_file = Path(output_dir) / 'SCREEN_LGR.grdecl'
    print('output:', output_file)
    print('GaP casing rows:', len(gap_casing))
    assert output_file.exists()
print('WellClass to GaP grid checks passed')

## Visual quality control

In [ ]:
import matplotlib.pyplot as plt

mesh = builder.grid_refine.mesh_df
mid_j = int(mesh['j'].max() // 2)
mesh_slice = mesh.query('j == @mid_j').pivot(index='k', columns='i', values='PERMX')

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for _, row in well_frames.holes_df.iterrows():
    axes[0].fill_betweenx([row['top_msl'], row['bottom_msl']], -row['diameter_m'] / 2, row['diameter_m'] / 2, alpha=0.35, color='steelblue')
for _, row in well_frames.casings_df.iterrows():
    axes[0].plot([-row['diameter_m'] / 2, row['diameter_m'] / 2], [row['top_msl'], row['top_msl']], color='black')
    axes[0].plot([-row['diameter_m'] / 2, row['diameter_m'] / 2], [row['bottom_msl'], row['bottom_msl']], color='black')
axes[0].invert_yaxis()
axes[0].set_title('WellClass geometry delivered to GaP')
axes[0].set_xlabel('radius [m]')
axes[0].set_ylabel('depth [mMSL]')

image = axes[1].imshow(mesh_slice, aspect='auto', origin='upper')
axes[1].set_title('Generated LGR PERMX cross-section')
axes[1].set_xlabel('x cell index')
axes[1].set_ylabel('z cell index')
fig.colorbar(image, ax=axes[1], label='PERMX')
plt.show()